In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

data = pd.read_csv("kdd_sample.csv")

le = LabelEncoder()
data['class'] = le.fit_transform(data['class'])

normal = data[data['class'] == 0]
anomaly = data[data['class'] == 1]
m = min(len(normal), len(anomaly))

balanced = pd.concat([
    normal.sample(m, random_state=42),
    anomaly.sample(m, random_state=42)
])

X = balanced.drop('class', axis=1)
y = balanced['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Training Normal:", sum(y_train == 0))
print("Training Anomaly:", sum(y_train == 1))
print("Testing Normal:", sum(y_test == 0))
print("Testing Anomaly:", sum(y_test == 1))


Training Normal: 70
Training Anomaly: 70
Testing Normal: 30
Testing Anomaly: 30


In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

p = Perceptron(max_iter=1000, random_state=42)
p.fit(X_train_s, y_train)

pred = p.predict(X_test_s)
print("Perceptron Accuracy:", accuracy_score(y_test, pred))


Perceptron Accuracy: 0.9166666666666666


In [5]:
for k in [2, 3, 4]:
    print("k =", k)
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(X_train_s)
    df = pd.DataFrame({'cluster': labels, 'class': y_train.values})
    for c in range(k):
        d = df[df['cluster'] == c]
        print(
            "Cluster", c,
            "Normal %:", (d['class'] == 0).mean() * 100,
            "Anomaly %:", (d['class'] == 1).mean() * 100
        )


k = 2
Cluster 0 Normal %: 16.666666666666664 Anomaly %: 83.33333333333334
Cluster 1 Normal %: 100.0 Anomaly %: 0.0
k = 3
Cluster 0 Normal %: 16.0 Anomaly %: 84.0
Cluster 1 Normal %: 22.22222222222222 Anomaly %: 77.77777777777779
Cluster 2 Normal %: 100.0 Anomaly %: 0.0
k = 4
Cluster 0 Normal %: 89.47368421052632 Anomaly %: 10.526315789473683
Cluster 1 Normal %: 25.0 Anomaly %: 75.0
Cluster 2 Normal %: 10.144927536231885 Anomaly %: 89.85507246376811
Cluster 3 Normal %: 100.0 Anomaly %: 0.0


In [6]:
k_opt = 2
km = KMeans(n_clusters=k_opt, random_state=42)
km.fit(X_train_s)

centroids = km.cluster_centers_
labels = km.labels_

centroid_labels = []
for i in range(k_opt):
    centroid_labels.append(y_train[labels == i].mode()[0])

predictions = []
for x in X_train_s:
    d = np.linalg.norm(centroids - x, axis=1)
    predictions.append(centroid_labels[np.argmin(d)])

print("Centroid Classification Accuracy:", accuracy_score(y_train, predictions))


Centroid Classification Accuracy: 0.9
